Example of Maximum Empirical Likelihood Estimation

Phys 4730 F26

In [ ]:
%reset -f
import numpy as np
import scipy.optimize as SciOpt
import scipy.stats as SciStat
import scipy.special as SciSpec
from numpy import random as rnd
from numpy import linalg as LA
import matplotlib.pyplot as plt
from scipy.stats.distributions import chi2
import matplotlib

matplotlib.rcParams.update({'font.size': 18})

In [ ]:
### sample the Weibull distribution ###

Nsamples = 20000
Shape    =     2
rvvec    = rnd.weibull(Shape,Nsamples)

In [ ]:
### plot the results ###

fig = plt.figure()
plt.hist(rvvec,30,range=(0.0,3.0))
plt.show()

print('The ground truth mean is : %8.5f ' % SciSpec.gamma(1+1/Shape))
print('The mean of the data is  : %8.5f ' % np.mean(rvvec))

In [ ]:
### function to evaluate empirical likelihood ratio ###

def ELR(data,valu):
    
    # Define the estimating function
    g = data - valu
    
    # Set up the Lagrange multiplier condition for Lam
    constraintEQN = lambda L : sum(g/(1.0 + L*g)) 
    
    # Find the multiplier
    Lam = SciOpt.fsolve(constraintEQN,0)
    
    # return the Empirical Likelihood Ratio W
    W = 2.0*np.sum(np.log(1.0 + Lam*g))
    return W
#

In [ ]:
### some arrays we need ###

# Let's pick some reasonable range for possible values of the mean
x = np.linspace(np.mean(rvvec)-0.04, np.mean(rvvec)+0.04,100)

# y will hold ELR values 
y = np.zeros(len(x))

In [ ]:
### for every possible mean value in our range, compute ELR... ###

for ii in np.arange(0,len(x)):
    y[ii] = ELR(rvvec,x[ii]) 

### ...and make note of minimum ELR ###
    
cen = x[np.argmin(y)]
print("Minimum ELR:",cen)

In [ ]:
### now make a plot ###

fig = plt.figure(figsize=(8,6))

plt.plot(x,y,label='EL Ratio', linewidth=3)
plt.xlabel('Possible values of mean')
plt.ylabel('EL ratio')

plt.show()

In [ ]:
### From the curve above, we want to compute the ###
### range within which 95% of the time we expect ###
### "true mean" to fall.                         ###
###                                              ###
### For starters, add 95% C.I. line to the curve ###

fig = plt.figure(figsize=(8,6))

plt.plot(x,y,label='EL Ratio', linewidth=3)
plt.xlabel('Possible values of mean')
plt.ylabel('EL ratio')

ycut = np.ones(len(x))*chi2.ppf(0.975,df=1)                   # <=== add
plt.plot(x,ycut,c='r',label='95% C.I. cutoff', linewidth=3)   # <=== add 
plt.legend()                                                  # <=== add 

plt.show()

In [ ]:
### now find intersections of the curves ###

# Define cost function which sets target ELR based on confidence level
ELRiterator = lambda p : ELR(rvvec,p)-chi2.ppf(0.975, df=1)

# Use brent method to find high-side intersection point...
limHi = SciOpt.brentq(ELRiterator, cen, cen+0.04)

# ... and low-side intersection point
limLo = SciOpt.brentq(ELRiterator, cen-0.04, cen)

print('The ground truth mean is : %8.5f ' % SciSpec.gamma(1+1/Shape))
print('The mean of the data is  : %8.5f ' % np.mean(rvvec))
print('The 95 pct CI for MELE   : [%8.5f,%8.5f]' % (limLo,limHi))